[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/15-the-llama-block.ipynb)

# Module 15: The LLaMA Block

Companion notebook to [Module 15](https://transformers-and-math.drmhse.com/modules/15-the-llama-block) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — RoPE: only the distance survives (section 15.1)

In [ ]:
def rope(x, pos, base=10000.0):
    """Rotate neighbouring pairs of the last axis of x by pos × frequency."""
    d = x.shape[-1]
    theta = base ** (-torch.arange(0, d, 2, dtype=torch.float64) / d)
    ang = pos[:, None] * theta
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * torch.cos(ang) - x2 * torch.sin(ang)
    out[..., 1::2] = x1 * torch.sin(ang) + x2 * torch.cos(ang)
    return out

q, k = torch.randn(1, 64, dtype=torch.float64), torch.randn(1, 64, dtype=torch.float64)
for m, n in [(3, 0), (103, 100), (5003, 5000)]:
    s = (rope(q, torch.tensor([m.__float__()])) * rope(k, torch.tensor([n.__float__()]))).sum()
    print(f'positions ({m}, {n}), gap {m - n}: score {s.item():.10f}')

## Part 2 — Stretching RoPE (section 15.2)
Which pairs see angles they never saw in training? Those whose wavelength exceeds the training length.

In [ ]:
d, L_train, L_new = 128, 4096, 32768
lam = L_new / L_train
for name, base in [('original', 10000.0), ('NTK-aware', 10000.0 * lam ** (d / (d - 2)))]:
    theta = base ** (-torch.arange(0, d, 2, dtype=torch.float64) / d)
    print(f'{name:10} base {base:10,.0f}: slowest pair turns {L_new * theta[-1].item():.3f} rad by token {L_new:,}; '
          f'fastest pair speed {theta[0].item():.2f} rad/token')
theta0 = 10000.0 ** (-torch.arange(0, d, 2, dtype=torch.float64) / d)
print(f'in training (base 10,000) the slowest pair only reached {L_train * theta0[-1].item():.3f} rad')
print('NTK-aware scaling slows the slowest pair until 32k tokens reach the same angle, and leaves the fastest pair alone.')

## Part 3 — RMSNorm, SwiGLU and grouped-query attention (sections 15.3–15.5)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps, self.g = eps, nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.g

class SwiGLU(nn.Module):
    def __init__(self, d, dff):
        super().__init__()
        self.gate, self.up, self.down = nn.Linear(d, dff, bias=False), nn.Linear(d, dff, bias=False), nn.Linear(dff, d, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class GQA(nn.Module):
    def __init__(self, d, hq, hkv):
        super().__init__()
        self.hq, self.hkv, self.dh = hq, hkv, d // hq
        self.q = nn.Linear(d, d, bias=False)
        self.kv = nn.Linear(d, 2 * hkv * self.dh, bias=False)
        self.o = nn.Linear(d, d, bias=False)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.hq, self.dh)
        k, v = self.kv(x).view(B, T, 2, self.hkv, self.dh).unbind(2)
        pos = torch.arange(T, dtype=torch.float64)
        q, k = rope(q.double().transpose(1, 2), pos).float(), rope(k.double().transpose(1, 2), pos).float()
        v = v.transpose(1, 2)
        rep = self.hq // self.hkv                      # each key-value head serves `rep` query heads
        k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).reshape(B, T, -1))

class LlamaBlock(nn.Module):
    def __init__(self, d=256, hq=8, hkv=2):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn, self.ffn = GQA(d, hq, hkv), SwiGLU(d, int(8 * d / 3))
    def forward(self, x):
        x = x + self.attn(self.n1(x))                 # Pre-norm: the residual stream itself is never normalized
        return x + self.ffn(self.n2(x))

block = LlamaBlock()
print('output shape:', tuple(block(torch.randn(2, 10, 256)).shape))
params = lambda m: sum(p.numel() for p in m.parameters())
print('SwiGLU (8/3 width) parameters:', params(block.ffn), ' vs GELU FFN at 4d:', 2 * 256 * 1024)
print('KV projection parameters with 2 KV heads:', params(block.attn.kv), ' with 8:', 2 * 256 * 256)

## Part 4 — The KV-cache saving (section 15.5)

In [ ]:
def kv_gib(layers, kv_heads, d_head, ctx, bytes_=2):
    return 2 * layers * kv_heads * d_head * ctx * bytes_ / 2 ** 30

for hkv in (32, 8, 1):
    print(f'LLaMA-3-8B shape, {hkv:2} KV heads, 8,192 tokens: {kv_gib(32, hkv, 128, 8192):.3f} GiB per sequence')

## Your turn
1. Replace RoPE in `GQA` with nothing, and show that shifting all positions by 100 now changes the scores. (It shouldn't change them with RoPE.)
2. Set `hkv=8` (equal to `hq`). Confirm that GQA then behaves exactly like standard multi-head attention.